# FirstAI / Orbit — independent Colab T4 training
Build sourced datasets, train FirstGPT from random initialization, instruction-tune Orbit, then download its checkpoint. No Drive mount or manual dataset uploads. Colab storage is temporary.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO=Path('/content/FirstAI')
if REPO.exists(): subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','https://github.com/llakshyaa-rgb/FirstAI.git',str(REPO)],check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO))
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda)
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, reconnect, and rerun.')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
subprocess.run([sys.executable,'-m','pip','install','-q','datasets>=3.0'],check=True)


## 1. Download and prepare real data
Downloads public training splits and retrieves fresh multi-domain public records. Applied examples are derived from retrieved values and retain provenance. Dataset licenses and caveats are recorded. This stage prepares data; it does not train weights.

In [ ]:
subprocess.run([sys.executable,'build_all_data.py','--country','IND','--city','New Delhi','--max-per-dataset','3000','--max-base-chars','5000000'],check=True)
import json
for f in ['data/capabilities/manifest.json','data/instructions_balanced.manifest.json']:
 p=Path(f)
 if p.exists(): print('\n'+f, json.dumps(json.loads(p.read_text()),indent=2)[:10000])


## 2. Train FirstGPT from scratch
This is a small experimental model, not a pretrained foundation model. A few million characters and a few thousand steps will not produce modern assistant-level capabilities; scale data and compute carefully after evaluating.

In [ ]:
subprocess.run([sys.executable,'train.py','--data','data/input.txt','--out','checkpoints/firstgpt-real.pt','--steps',os.environ.get('FIRSTAI_BASE_STEPS','3500'),'--eval-interval','200','--checkpoint-interval','500','--batch-size','4','--grad-accum-steps','4','--block-size','1024','--layers','6','--heads','8','--width','256','--vocab-size','512','--amp'],check=True)


## 3. Instruction-tune on practical tasks
Uses the balanced sourced examples for maths, science, reading, coding and applied real-data interpretation. Live retrieval remains necessary for changing facts.

In [ ]:
subprocess.run([sys.executable,'finetune_instructions.py','--checkpoint','checkpoints/firstgpt-real.best.pt','--data','data/instructions_balanced.jsonl','--out','checkpoints/orbit-instruct.pt','--steps',os.environ.get('FIRSTAI_SFT_STEPS','1800'),'--batch-size','4','--eval-interval','100'],check=True)
print('Orbit checkpoint bytes:',Path('checkpoints/orbit-instruct.pt').stat().st_size)


## 4. Download both checkpoints
Download before disconnecting; Colab's temporary disk is erased when the runtime ends.

In [ ]:
from google.colab import files
files.download('checkpoints/firstgpt-real.best.pt')
files.download('checkpoints/orbit-instruct.pt')
